# Baseline image classification — Softmax Regression + MLP

This notebook implements the **baseline** part of the assignment:
- load the **CIFAR-10** dataset
- preprocess data
- flatten images for vector-based models
- implement **Softmax regression**
- implement **MLP** with at least 2 linear layers
- write a custom **training loop**
- train and evaluate both models
- log **loss**, **accuracy**, and **training time**
- export results to **JSON** and **CSV**
- plot **loss/accuracy** curves

> Recommended workflow:
> 1. Run all setup cells
> 3. Train Softmax Regression
> 4. Train MLP
> 5. Export results and figures


In [ ]:
# If needed, uncomment and run:
# !pip install torch torchvision matplotlib pandas


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


In [ ]:
CONFIG = {
    "dataset_name": "CIFAR10",
    "data_root": "./data",
    "batch_size": 128,
    "num_workers": 2,
    "epochs": 10,
    "learning_rate_softmax": 1e-3,
    "learning_rate_mlp": 1e-3,
    "weight_decay": 0.0,
    "hidden_dims": [512, 256],  # MLP with 3 linear layers total
    "seed": 42,
    "output_dir": "./outputs_baseline_cifar10",
}

assert CONFIG["dataset_name"] == "CIFAR10", "This notebook is configured for CIFAR-10 only."

OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG


In [ ]:
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


In [ ]:
def get_dataset_and_loaders(config):
    data_root = config["data_root"]
    batch_size = config["batch_size"]
    num_workers = config["num_workers"]

    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    in_channels, height, width, num_classes = 3, 32, 32, 10

    train_dataset = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=True,
        transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]),
    )
    test_dataset = datasets.CIFAR10(
        root=data_root,
        train=False,
        download=True,
        transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    meta = {
        "dataset_name": "CIFAR10",
        "mean": mean,
        "std": std,
        "in_channels": in_channels,
        "height": height,
        "width": width,
        "input_dim": in_channels * height * width,
        "num_classes": num_classes,
        "class_names": train_dataset.classes,
    }
    return train_dataset, test_dataset, train_loader, test_loader, meta


In [ ]:
train_dataset, test_dataset, train_loader, test_loader, META = get_dataset_and_loaders(CONFIG)

print("Dataset:", META["dataset_name"])
print("Number of training samples:", len(train_dataset))
print("Number of test samples:", len(test_dataset))
print("Classes:", META["class_names"])


In [ ]:
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Flattened input dimension:", META["input_dim"])


## Model definitions

Both baseline models operate on **flattened CIFAR-10 images**.

- **Softmax Regression**: one linear layer from input dimension to class logits.
- **MLP**: multiple fully connected layers with ReLU activations.


In [ ]:
class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        logits = self.linear(x)
        return logits


class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], num_classes: int):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        logits = self.network(x)
        return logits


In [ ]:
softmax_model = SoftmaxRegression(META["input_dim"], META["num_classes"]).to(DEVICE)
mlp_model = MLP(META["input_dim"], CONFIG["hidden_dims"], META["num_classes"]).to(DEVICE)

softmax_model, mlp_model


## Training and evaluation utilities

This section uses a **custom training loop** with the required steps:
`forward -> loss -> backward -> optimizer.step()`


In [ ]:
def accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=1)
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return correct, total


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)                # forward
        loss = criterion(logits, targets)     # loss
        loss.backward()                       # backward
        optimizer.step()                      # optimizer step

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        correct, total = accuracy_from_logits(logits, targets)
        running_correct += correct
        running_total += total

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        correct, total = accuracy_from_logits(logits, targets)
        running_correct += correct
        running_total += total

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


def fit_model(model, train_loader, test_loader, epochs, lr, weight_decay, device, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "model": model_name,
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"[{model_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"
        )

    elapsed = time.perf_counter() - start_time

    summary = {
        "model": model_name,
        "train_acc": history["train_acc"][-1],
        "test_acc": history["test_acc"][-1],
        "train_loss": history["train_loss"][-1],
        "test_loss": history["test_loss"][-1],
        "train_time": elapsed,
        "dataset": META["dataset_name"],
        "batch_size": CONFIG["batch_size"],
        "epochs": epochs,
        "input_dim": META["input_dim"],
    }
    return history, summary


## Train Softmax Regression

In [ ]:
set_seed(CONFIG["seed"])
softmax_model = SoftmaxRegression(META["input_dim"], META["num_classes"]).to(DEVICE)

softmax_history, softmax_summary = fit_model(
    model=softmax_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=CONFIG["epochs"],
    lr=CONFIG["learning_rate_softmax"],
    weight_decay=CONFIG["weight_decay"],
    device=DEVICE,
    model_name="SoftmaxRegression",
)

softmax_summary


## Train MLP

In [ ]:
set_seed(CONFIG["seed"])
mlp_model = MLP(META["input_dim"], CONFIG["hidden_dims"], META["num_classes"]).to(DEVICE)

mlp_history, mlp_summary = fit_model(
    model=mlp_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=CONFIG["epochs"],
    lr=CONFIG["learning_rate_mlp"],
    weight_decay=CONFIG["weight_decay"],
    device=DEVICE,
    model_name="MLP",
)

mlp_summary


## Save model weights

In [ ]:
torch.save(softmax_model.state_dict(), OUTPUT_DIR / "softmax_regression.pt")
torch.save(mlp_model.state_dict(), OUTPUT_DIR / "mlp.pt")

print("Saved:", OUTPUT_DIR / "softmax_regression.pt")
print("Saved:", OUTPUT_DIR / "mlp.pt")


## Export results

This cell exports:
- individual JSON summaries
- combined JSON list
- CSV comparison table
- per-epoch history table


In [ ]:
results = [softmax_summary, mlp_summary]

with open(OUTPUT_DIR / "softmax_summary.json", "w", encoding="utf-8") as f:
    json.dump(softmax_summary, f, indent=2)

with open(OUTPUT_DIR / "mlp_summary.json", "w", encoding="utf-8") as f:
    json.dump(mlp_summary, f, indent=2)

with open(OUTPUT_DIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "baseline_results.csv", index=False)

history_rows = []
for epoch_idx in range(CONFIG["epochs"]):
    history_rows.append({
        "model": "SoftmaxRegression",
        "epoch": epoch_idx + 1,
        "train_loss": softmax_history["train_loss"][epoch_idx],
        "train_acc": softmax_history["train_acc"][epoch_idx],
        "test_loss": softmax_history["test_loss"][epoch_idx],
        "test_acc": softmax_history["test_acc"][epoch_idx],
    })
    history_rows.append({
        "model": "MLP",
        "epoch": epoch_idx + 1,
        "train_loss": mlp_history["train_loss"][epoch_idx],
        "train_acc": mlp_history["train_acc"][epoch_idx],
        "test_loss": mlp_history["test_loss"][epoch_idx],
        "test_acc": mlp_history["test_acc"][epoch_idx],
    })

history_df = pd.DataFrame(history_rows)
history_df.to_csv(OUTPUT_DIR / "baseline_history.csv", index=False)

results_df


## Plot learning curves

In [ ]:
def plot_metric(histories, metric_key, ylabel, title, save_path):
    plt.figure(figsize=(8, 5))
    for history in histories:
        plt.plot(history[metric_key], marker="o", label=f"{history['model']} - {metric_key}")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

plot_metric(
    [softmax_history, mlp_history],
    metric_key="train_loss",
    ylabel="Loss",
    title="Training Loss by Epoch",
    save_path=OUTPUT_DIR / "train_loss_curve.png",
)

plot_metric(
    [softmax_history, mlp_history],
    metric_key="test_loss",
    ylabel="Loss",
    title="Test Loss by Epoch",
    save_path=OUTPUT_DIR / "test_loss_curve.png",
)

plot_metric(
    [softmax_history, mlp_history],
    metric_key="train_acc",
    ylabel="Accuracy",
    title="Training Accuracy by Epoch",
    save_path=OUTPUT_DIR / "train_acc_curve.png",
)

plot_metric(
    [softmax_history, mlp_history],
    metric_key="test_acc",
    ylabel="Accuracy",
    title="Test Accuracy by Epoch",
    save_path=OUTPUT_DIR / "test_acc_curve.png",
)


## Quick comparison and interpretation

In [ ]:
results_df.sort_values(by="test_acc", ascending=False)


In [ ]:
best_model = results_df.sort_values(by="test_acc", ascending=False).iloc[0]
print(
    f"Best baseline model on {META['dataset_name']}: {best_model['model']} "
    f"with test_acc={best_model['test_acc']:.4f}"
)


## Suggested text for the report

You can adapt the following interpretation after training:

- Softmax Regression is a linear classifier and serves as the simplest baseline.
- MLP adds nonlinear hidden layers, so it usually learns more expressive decision boundaries.
- If MLP outperforms Softmax Regression, this suggests that nonlinear feature transformation is useful even before moving to CNN or Transformer-based models.
- The exported JSON/CSV files can be merged later with results from CNN, ViT, and other team members' models.


## Output checklist

After a successful run, the folder `outputs_baseline/` should contain:
- `softmax_regression.pt`
- `mlp.pt`
- `softmax_summary.json`
- `mlp_summary.json`
- `baseline_results.json`
- `baseline_results.csv`
- `baseline_history.csv`
- `train_loss_curve.png`
- `test_loss_curve.png`
- `train_acc_curve.png`
- `test_acc_curve.png`
